# 01 · Data download (raster track)

Downloads the rasters for the AOI + 1 km buffer (execution plan A1) and shows quick-looks:

- ESRI IO LULC v02, every year from `years.baseline` to `years.latest` (decision D2)
- ESA WorldCover 2021, cross-check only
- Copernicus DEM GLO-30

Reruns skip files whose checksum matches `data/manifest.json`. OSM layers are downloaded by `src.download.osm` (Harsh's track).

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import rasterio
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

from src.config import load_config
from src.download.dem import download_dem
from src.download.lulc import download_esri_lulc, download_worldcover
from src.io_utils import Manifest, setup_logging

cfg = load_config()
setup_logging(cfg, "download")
print(cfg["aoi"]["name"], cfg.crs.to_string(), f"{cfg.aoi_area_km2:.0f} km²")

## Download (skips anything already downloaded)

In [ ]:
esri = download_esri_lulc(cfg)
wc = download_worldcover(cfg)
dem = download_dem(cfg)

manifest = Manifest.for_config(cfg)
for key, e in sorted(manifest.entries.items()):
    print(f"{key:40s} {e['dataset']:24s} {e.get('year') or '':>5} {e['shape']} {e['crs']:>10} {e['size_bytes'] / 1e6:5.1f} MB")

## ESRI LULC: baseline vs latest

In [ ]:
ESRI = {1: ("Water", "#419bdf"), 2: ("Trees", "#397d49"), 4: ("Flooded veg.", "#7a87c6"),
        5: ("Crops", "#e49635"), 7: ("Built area", "#c4281b"), 8: ("Bare ground", "#a59b8f"),
        9: ("Snow/ice", "#a8ebff"), 10: ("Clouds", "#616161"), 11: ("Rangeland", "#e3e2c3")}
codes = sorted(ESRI)
lut = np.zeros(256, np.uint8)
for k, c in enumerate(codes):
    lut[c] = k + 1
cmap = ListedColormap(["white"] + [ESRI[c][1] for c in codes])


def show_lulc(path, ax, title):
    with rasterio.open(path) as ds:
        arr = ds.read(1)
        ext = (ds.bounds.left, ds.bounds.right, ds.bounds.bottom, ds.bounds.top)
        aoi = cfg.aoi.to_crs(ds.crs)
    ax.imshow(lut[arr], cmap=cmap, vmin=0, vmax=len(codes), extent=ext, interpolation="nearest")
    aoi.boundary.plot(ax=ax, color="black", lw=1)
    ax.set_title(title)
    ax.set_axis_off()
    return arr


years = cfg["years"]
fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))
a0 = show_lulc(esri[0], axes[0], f"ESRI LULC {years['baseline']}")
a1 = show_lulc(esri[-1], axes[1], f"ESRI LULC {years['latest']}")
present = [c for c in codes if (a0 == c).mean() > 0.001 or (a1 == c).mean() > 0.001]
fig.legend(handles=[Patch(color=ESRI[c][1], label=ESRI[c][0]) for c in present], loc="lower center", ncol=len(present))
plt.tight_layout(rect=(0, 0.06, 1, 1))

## Class shares per year and persistent growth

Persistent growth = non-built in baseline **and** baseline_confirm, built in latest_confirm **and** latest (decision D3).

In [ ]:
shares = {}
stack = {}
for p in esri:
    y = int(p.stem.split("_")[-1])
    with rasterio.open(p) as ds:
        stack[y] = ds.read(1)
    a = stack[y][stack[y] != 0]
    shares[y] = {ESRI[c][0]: round(float((a == c).mean()) * 100, 1) for c in codes if (a == c).any()}
import pandas as pd

display(pd.DataFrame(shares).T)

B = {y: s == 7 for y, s in stack.items()}
new = ~B[years["baseline"]] & ~B[years["baseline_confirm"]] & B[years["latest_confirm"]] & B[years["latest"]]
raw = ~B[years["baseline"]] & B[years["latest"]]
print(f"raw new built: {raw.mean():.1%} of area; persistent: {new.mean():.1%} ({new.sum() / max(raw.sum(), 1):.0%} of raw)")

with rasterio.open(esri[0]) as ds:
    ext = (ds.bounds.left, ds.bounds.right, ds.bounds.bottom, ds.bounds.top)
    aoi = cfg.aoi.to_crs(ds.crs)
view = np.where(B[years["baseline"]], 1, 0) + np.where(new, 2, 0)
fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(view, cmap=ListedColormap(["#f2f2f2", "#9e9e9e", "#d7191c"]), vmin=0, vmax=2, extent=ext, interpolation="nearest")
aoi.boundary.plot(ax=ax, color="black", lw=1)
ax.legend(handles=[Patch(color="#9e9e9e", label=f"built {years['baseline']}"), Patch(color="#d7191c", label="persistent new built")], loc="lower left")
ax.set_title("Persistent growth")
ax.set_axis_off()

## WorldCover 2021 (cross-check) and DEM

In [ ]:
WC = {10: "#006400", 20: "#ffbb22", 30: "#ffff4c", 40: "#f096ff", 50: "#fa0000", 60: "#b4b4b4",
      70: "#f0f0f0", 80: "#0064c8", 90: "#0096a0", 95: "#00cf75", 100: "#fae6a0"}
wlut = np.zeros(256, np.uint8)
for k, c in enumerate(sorted(WC)):
    wlut[c] = k + 1
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
with rasterio.open(wc) as ds:
    axes[0].imshow(wlut[ds.read(1)], cmap=ListedColormap(["white"] + [WC[c] for c in sorted(WC)]), vmin=0, vmax=len(WC),
                   extent=(ds.bounds.left, ds.bounds.right, ds.bounds.bottom, ds.bounds.top), interpolation="nearest")
    cfg.aoi.to_crs(ds.crs).boundary.plot(ax=axes[0], color="black", lw=1)
axes[0].set_title("WorldCover 2021")
with rasterio.open(dem) as ds:
    z = np.ma.masked_equal(ds.read(1), ds.nodata)
    im = axes[1].imshow(z, cmap="terrain", extent=(ds.bounds.left, ds.bounds.right, ds.bounds.bottom, ds.bounds.top))
    cfg.aoi.to_crs(ds.crs).boundary.plot(ax=axes[1], color="black", lw=1)
fig.colorbar(im, ax=axes[1], label="elevation (m)")
axes[1].set_title("Copernicus DEM GLO-30")
for a in axes:
    a.set_axis_off()
plt.tight_layout()

## Checks

- Every raster covers the AOI + buffer, has a CRS and a nodata value.
- ESRI maps contain only documented class codes.
- Tree cover: ESRI reports far less than WorldCover here (much of Bannerghatta's scrub forest is *rangeland* in ESRI). Keep this in mind for the forest class in clustering (H4).

In [ ]:
from shapely.geometry import box

need = cfg.aoi_projected.buffer(cfg["aoi"]["buffer_m"] * 0.9)
for p in [*esri, wc, dem]:
    with rasterio.open(p) as ds:
        covers = box(*ds.bounds).contains(need.to_crs(ds.crs).iloc[0])
        extra = set(np.unique(ds.read(1)).tolist()) - set(ESRI) - {0} if p in esri else set()
        print(f"{p.name:22s} crs={ds.crs.to_epsg()} nodata={ds.nodata} covers_aoi_buffer={covers} unknown_classes={extra or '-'}")